# Ejercicio 2. Transformación y calidad de datos

## Objetivo

Limpiar los registros de ventas almacenados en `ventas_bronze` y generar una versión confiable llamada `ventas_silver`.

Aplicaré las siguientes reglas:

1. Eliminar registros idénticos duplicados.
2. Eliminar ventas sin producto.
3. Eliminar ventas con cantidad menor o igual a cero.
4. Eliminar ventas con precio unitario menor o igual a cero.
5. Calcular el valor total de cada venta.
6. Agrupar el valor vendido por categoría.

Por último, comprobaré que ejecutar nuevamente el notebook no genera duplicados en las tablas de salida.


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

catalogo = "workspace"
esquema = "ingenieria_datos_luismiguel"

tabla_bronze = f"{catalogo}.{esquema}.ventas_bronze"
tabla_silver = f"{catalogo}.{esquema}.ventas_silver"
tabla_resumen = f"{catalogo}.{esquema}.ventas_por_categoria"

df_bronze = spark.table(tabla_bronze)

print("Registros originales:", df_bronze.count())
display(df_bronze)


Registros originales: 15


id_venta,fecha,producto,categoria,cantidad,precio_unitario
1,2026-09-03,The Legend of Zelda: Link's Awakening Remake,Videojuegos,2,249900.0
2,2026-09-08,Minecraft,Videojuegos,4,119900.0
3,2026-09-15,Control Xbox Series,Accesorios,3,289900.0
4,2026-09-22,PlayStation 5,Consolas,1,2599900.0
5,2026-10-02,GTA VI - Preventa,Preventas,2,349900.0
6,2026-10-06,Audifonos HyperX Cloud II,Accesorios,3,329900.0
7,2026-10-10,Nintendo Switch 2,Consolas,2,2199900.0
8,2026-10-14,EA Sports FC 26,Videojuegos,3,299900.0
9,2026-10-20,Teclado mecanico Redragon,Accesorios,4,189900.0
10,2026-10-28,Control DualSense PS5,Accesorios,2,329900.0


## 2.1. Limpieza de registros

Se utiliza `dropDuplicates()` para eliminar las filas idénticas.

Posteriormente, se aplican filtros para descartar registros sin producto, con cantidades inválidas o con precios menores o iguales a cero.

El conjunto original contiene 15 registros y se espera obtener 12 registros válidos después de la limpieza.


In [0]:
df_silver = (
    df_bronze
    .dropDuplicates()
    .filter(
        col("producto").isNotNull() &
        (col("cantidad") > 0) &
        (col("precio_unitario") > 0)
    )
    .withColumn(
        "total_venta",
        col("cantidad") * col("precio_unitario")
    )
)

print("Registros antes:", df_bronze.count())
print("Registros después:", df_silver.count())

display(df_silver.orderBy("id_venta"))


Registros antes: 15
Registros después: 12


id_venta,fecha,producto,categoria,cantidad,precio_unitario,total_venta
1,2026-09-03,The Legend of Zelda: Link's Awakening Remake,Videojuegos,2,249900.0,499800.0
2,2026-09-08,Minecraft,Videojuegos,4,119900.0,479600.0
3,2026-09-15,Control Xbox Series,Accesorios,3,289900.0,869700.0
4,2026-09-22,PlayStation 5,Consolas,1,2599900.0,2599900.0
5,2026-10-02,GTA VI - Preventa,Preventas,2,349900.0,699800.0
6,2026-10-06,Audifonos HyperX Cloud II,Accesorios,3,329900.0,989700.0
7,2026-10-10,Nintendo Switch 2,Consolas,2,2199900.0,4399800.0
8,2026-10-14,EA Sports FC 26,Videojuegos,3,299900.0,899700.0
9,2026-10-20,Teclado mecanico Redragon,Accesorios,4,189900.0,759600.0
10,2026-10-28,Control DualSense PS5,Accesorios,2,329900.0,659800.0


## 2.2. Almacenamiento de ventas limpias

Los registros válidos se guardan en una nueva tabla Delta llamada `ventas_silver`.

Se utiliza el modo `overwrite` para reemplazar el contenido anterior y evitar la acumulación de datos al ejecutar nuevamente el proceso.


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabla_silver)

print("Tabla Silver creada correctamente")
print("Registros guardados:", spark.table(tabla_silver).count())


Tabla Silver creada correctamente
Registros guardados: 12


## 2.3. Resumen de ventas por categoría

Se agrupan los registros limpios por categoría y se calcula la suma de `total_venta`.

Este resumen permite identificar cuánto dinero generó cada categoría de la tienda de videojuegos.


In [0]:
df_resumen = (
    spark.table(tabla_silver)
    .groupBy("categoria")
    .agg(
        spark_sum("total_venta").alias("total_vendido")
    )
)

df_resumen.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabla_resumen)

display(df_resumen.orderBy("categoria"))


categoria,total_vendido
Accesorios,4028500.0
Consolas,6999700.0
Preventas,699800.0
Videojuegos,2398900.0


## 2.4. Validación de resultados

Para comprobar que el proceso es repetible, se ejecuta nuevamente el notebook completo.

El uso de `overwrite` permite reemplazar las tablas de salida sin acumular registros duplicados.

Se verifican los siguientes resultados:

- `ventas_bronze`: 15 registros originales.
- `ventas_silver`: 12 registros válidos.
- `ventas_por_categoria`: resumen agrupado por categoría.

De esta manera, el proceso mantiene resultados consistentes al ejecutarse varias veces.


In [0]:
print("Bronze:", spark.table(tabla_bronze).count())
print("Silver:", spark.table(tabla_silver).count())
print("Categorías:", spark.table(tabla_resumen).count())

display(spark.table(tabla_resumen).orderBy("categoria"))


Bronze: 15
Silver: 12
Categorías: 4


categoria,total_vendido
Accesorios,4028500.0
Consolas,6999700.0
Preventas,699800.0
Videojuegos,2398900.0
